# HelmX Crash Detection — Evaluation

Final metrics on the **held-out test split** (never touched during training or threshold
calibration), an ablation comparing the reconstruction-only / classifier-only / hybrid decision
rules, and a simulation of the actual deployed alert logic (3 consecutive windows + cooldown)
to estimate real-world alert frequency rather than a raw per-window false-positive rate.

Run `crash-detection-model.ipynb` first to produce `models/helmx_crash_model.keras`.

In [ ]:
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf

from src import config, io_loaders, preprocess, manifest, normalize, evaluate
from src import model as model_lib  # noqa: F401 (registers the custom reconstruction loss for load_model)

In [ ]:
def plot_roc_pr(y_true, scores, name):
    curves = evaluate.roc_pr_curves(y_true, scores)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(curves["fpr"], curves["tpr"])
    axes[0].plot([0, 1], [0, 1], "--", color="gray")
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate")
    axes[0].set_title(f"{name} ROC")

    axes[1].plot(curves["recall"], curves["precision"])
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].set_title(f"{name} PR")

    fig.tight_layout()
    fig.savefig(config.RESULTS_DIR / f"{name}_roc_pr.png", dpi=150)
    plt.show()


def plot_confusion_matrix(cm, name):
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i][j]), ha="center", va="center")
    ax.set_xticks([0, 1], ["Normal", "Crash"])
    ax.set_yticks([0, 1], ["Normal", "Crash"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"{name} Confusion Matrix")
    fig.tight_layout()
    fig.savefig(config.RESULTS_DIR / f"{name}_confusion_matrix.png", dpi=150)
    plt.show()

## Load the held-out test split and run inference

In [ ]:
config.RESULTS_DIR.mkdir(exist_ok=True, parents=True)

sessions = preprocess.canonicalize_all(io_loaders.load_all_sessions())
split_of = manifest.assign_splits(sessions)
test_ds = manifest.build_dataset(sessions, split_of, "test")
print(f"Test set: {len(test_ds.y)} windows "
      f"({int((test_ds.y == 0).sum())} normal, {int((test_ds.y == 1).sum())} anomaly)")

normalizer = normalize.Normalizer.load(config.MODELS_DIR / "normalizer.npz")
X_seq, X_aux = normalizer.transform(test_ds.X_seq, test_ds.X_aux)

model = tf.keras.models.load_model(config.MODELS_DIR / "helmx_crash_model.keras")
preds = model.predict({"seq": X_seq, "aux": X_aux}, verbose=0)
crash_prob, recon_error = preds["crash_prob"].ravel(), preds["recon_error"].ravel()

with open(config.MODELS_DIR / "thresholds.json") as f:
    thresholds = json.load(f)

window_seconds = config.WINDOW_SAMPLES / config.TARGET_RATE_HZ

## ROC / PR curves

In [ ]:
plot_roc_pr(test_ds.y, recon_error, "reconstruction")
plot_roc_pr(test_ds.y, crash_prob, "classifier")

## Ablation: reconstruction-only vs. classifier at each operating point vs. hybrid

In [ ]:
results = {}
for variant, scores, threshold in [
    ("reconstruction_only", recon_error, thresholds["recon_error"]),
    ("classifier_only_balanced", crash_prob, thresholds["classifier_prob_balanced"]),
    ("classifier_only_moderate_recall", crash_prob, thresholds["classifier_prob_moderate_recall"]),
    ("classifier_only_high_recall", crash_prob, thresholds["classifier_prob_high_recall"]),
]:
    m = evaluate.summarize(test_ds.y, scores, threshold, window_seconds=window_seconds)
    results[variant] = m
    plot_confusion_matrix(m["confusion_matrix"], variant)

hybrid_pred = (
    (crash_prob >= thresholds["classifier_prob"]) | (recon_error >= thresholds["recon_error"])
).astype(int)
m = evaluate.summarize_predictions(test_ds.y, hybrid_pred, window_seconds=window_seconds)
results["hybrid"] = m
plot_confusion_matrix(m["confusion_matrix"], "hybrid")

## Simulated deployed-alert rate

Window-level false-positive rate overstates real alarm frequency: `deploy/crash_monitor.py` only fires after 3 consecutive over-threshold windows, then goes quiet for a cooldown. This replays that rule over the ordered test windows per session to estimate actual alert events/hour and session-level recall.

In [ ]:
deployed = {}
for variant, threshold_key in [
    ("classifier_only_balanced", "classifier_prob_balanced"),
    ("classifier_only_moderate_recall", "classifier_prob_moderate_recall"),
    ("classifier_only_high_recall", "classifier_prob_high_recall"),
]:
    deployed[variant] = evaluate.simulate_deployed_alerts(
        test_ds.y, crash_prob, test_ds.session_id, thresholds[threshold_key],
        window_seconds=window_seconds,
    )
results["deployed_alert_simulation"] = deployed

print(json.dumps(results, indent=2))
evaluate.save_metrics(results, "test_metrics")